In [ ]:
{
 "cells": [
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "# Pranav Sai Portfolio Chatbot (RAG + Gradio)\n",
    "\n",
    "This notebook demonstrates how to run a multi-file Python project (`rag_engine.py` and `app.py`) in a single Colab environment, integrating the RAG logic with a Gradio web interface. The RAG uses the `google/flan-t5-large` model for improved accuracy.\n",
    "\n",
    "**Instructions:**\n",
    "1.  **Run Cell 1:** Installs dependencies.\n",
    "2.  **Run Cell 2:** Writes the modular Python files to the virtual machine's filesystem.\n",
    "3.  **Run Cell 3:** Executes the main `app.py` script to launch the Gradio interface. Click the public URL to access the chatbot."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Cell 1: Install Dependencies\n",
    "!pip install transformers langchain langchain-community sentence-transformers accelerate torch gradio\n",
    "\n",
    "print(\"Dependencies installed successfully.\")"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Cell 2: Write Python Files to Disk\n",
    "\n",
    "# Create data directory\n",
    "!mkdir -p data\n",
    "\n",
    "### FILE: rag_engine.py (The RAG Logic)\n",
    "%%writefile rag_engine.py\n",
    "# ------------------- imports ---------------------\n",
    "import os\n",
    "import time\n",
    "from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, pipeline\n",
    "from langchain.text_splitter import RecursiveCharacterTextSplitter\n",
    "from langchain_community.document_loaders import TextLoader\n",
    "from langchain_community.vectorstores import Chroma\n",
    "from langchain_community.embeddings import HuggingFaceEmbeddings\n",
    "from langchain.llms import HuggingFacePipeline\n",
    "from langchain.prompts import PromptTemplate\n",
    "from langchain.chains import LLMChain\n",
    "\n",
    "# ---------------- Constants & Global Setup ----------------\n",
    "DATA_DIR = \"data\"\n",
    "PORTFOLIO_FILE = os.path.join(DATA_DIR, \"portfolio.txt\")\n",
    "DB_DIR = \"db\"\n",
    "MODEL_ID = \"google/flan-t5-large\" # Using the larger model for better RAG performance\n",
    "\n",
    "PORTFOLIO_CONTENT = \"\"\"\n",
    "## Chaitanya Pranav Sai Kodamasimham - Professional Portfolio\n",
    "\n",
    "### Contact Information\n",
    "Email: chaitanya.kodamasimham@example.com\n",
    "LinkedIn: /in/chaitanya-pranav-sai-k\n",
    "\n",
    "### Summary\n",
    "Highly motivated and results-oriented Software Engineer with 4 years of experience specializing in full-stack development using React, Node.js, and AWS serverless technologies. Proven ability to lead projects from concept to deployment, focusing on scalable and maintainable code architecture.\n",
    "\n",
    "### Skills\n",
    "- **Programming Languages:** Python, JavaScript (ES6+), TypeScript, Java\n",
    "- **Frontend:** React, Redux, Tailwind CSS, Next.js, HTML5, CSS3\n",
    "- **Backend/APIs:** Node.js, Express.js, GraphQL, REST\n",
    "- **Databases:** PostgreSQL, MongoDB, Redis, DynamoDB\n",
    "- **Cloud/DevOps:** AWS (Lambda, S3, RDS, API Gateway, EC2), Docker, CI/CD (GitHub Actions)\n",
    "- **Other:** Git, Agile Scrum, Unit Testing (Jest, Mocha)\n",
    "\n",
    "### Professional Experience\n",
    "\n",
    "**Senior Software Engineer | TechCorp Innovations | 2022 - Present**\n",
    "- Led the migration of a monolithic API service to a serverless architecture on AWS Lambda, resulting in a 40% reduction in operational costs.\n",
    "- Developed a real-time analytics dashboard using React and WebSockets, handling over 10,000 concurrent users.\n",
    "- Mentored junior engineers on best practices for React component lifecycle and state management with Redux.\n",
    "\n",
    "**Software Developer | Digital Solutions Inc. | 2020 - 2022**\n",
    "- Designed and implemented a secure OAuth 2.0 authentication system for the primary customer portal.\n",
    "- Improved application load time by 35% through optimizing database queries and implementing effective caching strategies using Redis.\n",
    "\n",
    "### Education\n",
    "**Master of Science in Computer Science**\n",
    "Carnegie Mellon University | Pittsburgh, PA | 2018 - 2020\n",
    "**Bachelor of Technology in Information Technology**\n",
    "JNTU Hyderabad | Hyderabad, India | 2014 - 2018\n",
    "\"\"\"\n",
    "\n",
    "# Ensure data directory and file exist\n",
    "os.makedirs(DATA_DIR, exist_ok=True)\n",
    "with open(PORTFOLIO_FILE, \"w\", encoding=\"utf-8\") as f:\n",
    "    f.write(PORTFOLIO_CONTENT)\n",
    "\n",
    "# ---------------- Document Loading & DB Building ----------------\n",
    "def load_documents():\n",
    "    \"\"\"Load portfolio.txt into LangChain documents.\"\"\"\n",
    "    loader = TextLoader(PORTFOLIO_FILE, encoding=\"utf-8\")\n",
    "    return loader.load()\n",
    "\n",
    "def build_vector_db():\n",
    "    \"\"\"Build or load the vector database.\"\"\"\n",
    "    print(\"Building or loading vector database...\")\n",
    "    docs = load_documents()\n",
    "    splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=200)\n",
    "    chunks = splitter.split_documents(docs)\n",
    "    embeddings = HuggingFaceEmbeddings(model_name=\"sentence-transformers/all-MiniLM-L6-v2\")\n",
    "\n",
    "    db = Chroma.from_documents(chunks, embeddings, persist_directory=DB_DIR)\n",
    "    db.persist()\n",
    "    print(f\"Vector database built with {len(chunks)} chunks.\")\n",
    "    return db\n",
    "\n",
    "# ---------------- FLAN-T5 LLM Setup ----------------\n",
    "def setup_flant5_llm():\n",
    "    \"\"\"Loads the FLAN-T5 model and sets up the LangChain pipeline.\"\"\"\n",
    "    print(f\"Loading FLAN-T5 model: {MODEL_ID} (This may take a moment)...  \")\n",
    "    start_time = time.time()\n",
    "    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)\n",
    "    model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_ID)\n",
    "\n",
    "    generator = pipeline(\n",
    "        \"text2text-generation\",\n",
    "        model=model,\n",
    "        tokenizer=tokenizer,\n",
    "        max_new_tokens=256,\n",
    "        temperature=0.1,\n",
    "        return_full_text=False\n",
    "    )\n",
    "    print(f\"Model loaded in {time.time() - start_time:.2f} seconds.\")\n",
    "    return HuggingFacePipeline(pipeline=generator)\n",
    "\n",
    "# ---------------- RAG Initialization (Runs when imported) ----------------\n",
    "\n",
    "# These variables hold the initialized RAG components\n",
    "if not os.path.exists(DB_DIR) or not os.listdir(DB_DIR):\n",
    "    db = build_vector_db()\n",
    "else:\n",
    "    print(\"Loading existing vector database...\")\n",
    "    embeddings = HuggingFaceEmbeddings(model_name=\"sentence-transformers/all-MiniLM-L6-v2\")\n",
    "    db = Chroma(persist_directory=DB_DIR, embedding_function=embeddings)\n",
    "\n",
    "retriever = db.as_retriever(search_kwargs={\"k\": 5})\n",
    "flant5_llm = setup_flant5_llm()\n",
    "\n",
    "RAG_PROMPT = \"\"\"\n",
    "You are an expert assistant for Chaitanya Pranav Sai Kodamasimham's portfolio.\n",
    "Answer the user's question using ONLY the provided context.\n",
    "- Give bullet points for skills or projects.\n",
    "- Summarize education briefly.\n",
    "- Do NOT invent information.\n",
    "\n",
    "Context:\n",
    "{context}\n",
    "\n",
    "Question:\n",
    "{question}\n",
    "\n",
    "Answer:\n",
    "\"\"\"\n",
    "prompt_template = PromptTemplate(template=RAG_PROMPT, input_variables=[\"context\", \"question\"])\n",
    "rag_chain = LLMChain(llm=flant5_llm, prompt=prompt_template)\n",
    "\n",
    "# ---------------- Exposed Function ----------------\n",
    "def ask_rag(question: str) -> str:\n",
    "    \"\"\"\n",
    "    Performs RAG query using the initialized components.\n",
    "    This is the function app.py will import and use.\n",
    "    \"\"\"\n",
    "    # print(f\"\\n--- Retrieving context for: {question} ---\")\n",
    "    \n",
    "    # 1. Retrieval\n",
    "    context_docs = retriever.get_relevant_documents(question)\n",
    "    context = \"\\n\\n\".join([d.page_content for d in context_docs])\n",
    "    \n",
    "    # 2. Generation Input\n",
    "    final_input = {\"context\": context, \"question\": question}\n",
    "\n",
    "    # 3. Generation (using the initialized chain)\n",
    "    answer = rag_chain.run(final_input)\n",
    "\n",
    "    # Clean and return answer\n",
    "    return answer.strip()\n",
    "\n",
    "\n",
    "### FILE: app.py (The Gradio Interface)\n",
    "%%writefile app.py\n",
    "import gradio as gr\n",
    "from typing import List, Tuple\n",
    "from rag_engine import ask_rag\n",
    "\n",
    "# ---------------- Chat Function ----------------\n",
    "def chat_with_rag(user_input: str, chat_history: List[Tuple[str, str]] = None):\n",
    "    if chat_history is None:\n",
    "        chat_history = []\n",
    "\n",
    "    if not user_input.strip():\n",
    "        # Handle empty input gracefully\n",
    "        return chat_history, \"\"\n",
    "\n",
    "    # Get answer from RAG\n",
    "    # Note: The model loading happens when rag_engine is imported, so this is fast.\n",
    "    answer = ask_rag(user_input)\n",
    "\n",
    "    # Append to history\n",
    "    chat_history.append((user_input, answer))\n",
    "\n",
    "    # Clear input box\n",
    "    return chat_history, \"\"\n",
    "\n",
    "# ---------------- Gradio Interface ----------------\n",
    "with gr.Blocks(theme=gr.themes.Soft(), title=\"Pranav Sai Portfolio Bot\") as demo:\n",
    "    gr.Markdown(\n",
    "        \"\"\"\n",
    "        <div style=\"text-align:center; padding:15px; background-color:#f0f4f8; border-radius:12px; margin-bottom:20px;\">\n",
    "            <h1 style=\"color:#3b82f6; font-size:2.5em; font-weight:700;\">Pranav Sai — Portfolio FAQ Bot</h1>\n",
    "            <p style=\"color:#4b5563;\">Ask about Pranav's skills, education, experience, or projects.</p>\n",
    "        </div>\n",
    "        \"\"\"\n",
    "    )\n",
    "\n",
    "    chatbot = gr.Chatbot(\n",
    "        label=\"Chat History\",\n",
    "        height=500,\n",
    "        # The image path is relative to the Gradio server, use a placeholder or public URL in Colab\n",
    "        avatar_images=[None, \"[https://avatars.githubusercontent.com/u/108990666?v=4](https://avatars.githubusercontent.com/u/108990666?v=4)\"],\n",
    "        show_copy_button=True\n",
    "    )\n",
    "\n",
    "    with gr.Row():\n",
    "        user_input = gr.Textbox(label=\"Ask a question\", placeholder=\"e.g., What are Pranav's skills?\", scale=4)\n",
    "        submit_btn = gr.Button(\"Send\", variant=\"primary\", scale=1)\n",
    "\n",
    "    clear_btn = gr.ClearButton([user_input, chatbot], value=\"Clear Chat\")\n",
    "\n",
    "    # Connect buttons\n",
    "    submit_btn.click(chat_with_rag, inputs=[user_input, chatbot], outputs=[chatbot, user_input])\n",
    "    user_input.submit(chat_with_rag, inputs=[user_input, chatbot], outputs=[chatbot, user_input])\n",
    "\n",
    "# NOTE: In Colab, we use share=True to generate a public URL.\n",
    "if __name__ == \"__main__\":\n",
    "    # server_name=\"127.0.0.1\" and server_port=7860 are often overridden in Colab, but we keep them as good practice.\n",
    "    demo.launch(share=True)\n",
    "print(\"Python files written. Ready to launch Gradio.\")\n"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Cell 3: Execute the App\n",
    "\n",
    "!python app.py"
   ]
  }
 ],
 "metadata": {
  "accelerator": "GPU",
  "colab": {
   "collapsed_sections": [],
   "name": "RAG Chatbot with Gradio Interface",
   "toc_visible": true
  },
  "kernelspec": {
   "display_name": "Python 3",
   "language": "python"
  },
  "language_info": {
   "codemirror_mode": {
    "name": "ipython",
    "version": 3
   },
   "file_extension": ".py",
   "mimetype": "text/x-python",
   "name": "python",
   "nbconvert_exporter": "python",
   "pygments_lexer": "ipython3",
   "version": "3.10.12"
  }
 },
 "nbformat": 4,
 "nbformat_minor": 0
}
